# Simulator Comparison Notebook

This notebook visualizes a quantum circuit and demonstrates that `simulator_own` reproduces the same statevector as the Aer simulator up to global phase.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from fp_qgpu.simulator import simulator_own
from fp_qgpu.simulator_mock import simulator_mock

## Build and Show Circuit

In [ ]:
qc = QuantumCircuit(3)
qc.h(0)
qc.cx(0, 1)
qc.cx(1, 2)

qc_trans = transpile(qc, basis_gates=["u", "cx"])
print(qc_trans.draw(output="text"))

## Verify `simulator_own` Against Aer

We compare Aer and the custom simulator on the same transpiled circuit.
The check accounts for global phase, so physically equivalent states are treated as equal.

In [ ]:
counts_aer, psi_aer = simulator_mock(qc_trans, shots=2048, seed=42)
psi_own = simulator_own(qc_trans)

idx = int(np.argmax(np.abs(psi_aer)))
phase = psi_aer[idx] / psi_own[idx]
same_state = np.allclose(psi_aer, psi_own * phase, atol=1e-12)

print("Aer counts (None expected here):", counts_aer)
print("Equivalent up to global phase:", same_state)

In [ ]:
assert same_state
print("Result: simulator_own matches Aer for this circuit.")